# SigLIP2 Furniture Complementary Retrieval

This notebook tests a simple complementary-product hypothesis:

1. Start with a product.
2. Keep candidates in the same room-level furniture category.
3. Exclude products from the query's exact leaf category.
4. Rank the remaining candidates by SigLIP2 similarity.

For example, a sofa can retrieve visually or textually similar living-room products, but other sofas are excluded. This is **taxonomy-constrained similarity**, not learned co-purchase complementarity. It is a useful baseline for inspecting whether the idea is promising.

## 1. Prerequisites

Select the `RecSystem — SigLIP2 (Python 3.11)` kernel.

This notebook reads the 500-product held-out catalog and the winning `multichunk64` embeddings created by `siglip2_furniture_text_strategy_benchmark.ipynb`. It does not retrain the model. If those local generated files are missing, run the benchmark notebook first.

In [ ]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps

SEED = 42
TOP_K = 8
MAX_PER_LEAF = 2
ROOM_COLUMN = "furniture_subcategory"
LEAF_COLUMN = "leaf_category"

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root. Open this notebook from the RecSystem repository.")

ROOT = find_project_root()
ARTIFACT_DIR = ROOT / "models" / "siglip2_furniture_text_strategy_benchmark"
CATALOG_PATH = ARTIFACT_DIR / "embedding_catalog.csv"
IMAGE_EMBEDDINGS_PATH = ARTIFACT_DIR / "multichunk64_image_embeddings.npy"
TEXT_EMBEDDINGS_PATH = ARTIFACT_DIR / "multichunk64_text_embeddings.npy"

print({"project_root": str(ROOT), "artifact_directory": str(ARTIFACT_DIR)})

In [ ]:
required_paths = [CATALOG_PATH, IMAGE_EMBEDDINGS_PATH, TEXT_EMBEDDINGS_PATH]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing generated benchmark artifacts:\n"
        + "\n".join(f"- {path}" for path in missing)
        + "\nRun siglip2_furniture_text_strategy_benchmark.ipynb first."
    )

catalog = pd.read_csv(CATALOG_PATH, dtype={"asin": "string"}).reset_index(drop=True)
image_embeddings = np.load(IMAGE_EMBEDDINGS_PATH).astype(np.float32)
text_embeddings = np.load(TEXT_EMBEDDINGS_PATH).astype(np.float32)

assert len(catalog) == len(image_embeddings) == len(text_embeddings)
assert catalog["asin"].is_unique
assert np.isfinite(image_embeddings).all() and np.isfinite(text_embeddings).all()
assert np.allclose(np.linalg.norm(image_embeddings, axis=1), 1.0, atol=1e-4)
assert np.allclose(np.linalg.norm(text_embeddings, axis=1), 1.0, atol=1e-4)

asin_to_index = pd.Series(catalog.index, index=catalog["asin"]).to_dict()
print({
    "catalog_rows": len(catalog),
    "embedding_dimension": image_embeddings.shape[1],
    "rooms": catalog[ROOM_COLUMN].nunique(),
    "leaf_categories": catalog[LEAF_COLUMN].nunique(),
})

## 2. Define the taxonomy constraint

We use `furniture_subcategory` as the room-level group. This is more reliable than blindly moving one literal path level upward because the hierarchy has variable depth:

- Sofa: `Furniture > Living Room Furniture > Sofas & Couches`
- Coffee table: `Furniture > Living Room Furniture > Tables > Coffee Tables`

For both products, the desired complement group is `Living Room Furniture`, even though it is not the immediate parent of `Coffee Tables`.

In [ ]:
room_summary = (
    catalog.groupby(ROOM_COLUMN)
    .agg(products=("asin", "size"), leaf_categories=(LEAF_COLUMN, "nunique"))
    .sort_values("products", ascending=False)
)
display(room_summary)

living_room_leaves = (
    catalog.loc[catalog[ROOM_COLUMN].eq("Living Room Furniture"), LEAF_COLUMN]
    .value_counts()
    .rename_axis("leaf_category")
    .to_frame("products")
)
display(living_room_leaves)

## 3. Complement retrieval

Available modes:

- `image_to_image`: visual similarity between the query and candidate images.
- `text_to_text`: similarity between the multi-chunk product descriptions.
- `image_to_text`: query image compared with candidate descriptions.
- `text_to_image`: query description compared with candidate images.

All embeddings are unit-normalized, so the dot product is cosine similarity. `max_per_leaf` is an optional diversification cap; set it to `None` for the raw ranking.

In [ ]:
MODE_SPACES = {
    "image_to_image": (image_embeddings, image_embeddings),
    "text_to_text": (text_embeddings, text_embeddings),
    "image_to_text": (image_embeddings, text_embeddings),
    "text_to_image": (text_embeddings, image_embeddings),
}

def product_index(asin):
    asin = str(asin)
    if asin not in asin_to_index:
        raise KeyError(f"ASIN {asin!r} is not in this 500-product catalog.")
    return int(asin_to_index[asin])

def complement_candidate_indices(query_index):
    query = catalog.iloc[query_index]
    room = query[ROOM_COLUMN]
    leaf = query[LEAF_COLUMN]
    if pd.isna(room) or room == "Unspecified":
        raise ValueError("The query has no usable room-level taxonomy.")
    mask = (
        catalog[ROOM_COLUMN].eq(room)
        & catalog[LEAF_COLUMN].ne(leaf)
        & catalog.index.to_series().ne(query_index)
    )
    indices = catalog.index[mask].to_numpy(dtype=int)
    if not len(indices):
        raise ValueError(f"No cross-leaf candidates are available in {room!r}.")
    return indices

def retrieve_complements(asin, mode="image_to_image", top_k=TOP_K, max_per_leaf=MAX_PER_LEAF):
    if mode not in MODE_SPACES:
        raise ValueError(f"Unknown mode {mode!r}. Choose from {list(MODE_SPACES)}.")
    query_index = product_index(asin)
    candidate_indices = complement_candidate_indices(query_index)
    query_space, candidate_space = MODE_SPACES[mode]
    scores = candidate_space[candidate_indices] @ query_space[query_index]
    ordered = candidate_indices[np.argsort(-scores, kind="stable")]
    score_by_index = dict(zip(candidate_indices.tolist(), scores.tolist()))

    selected, leaf_counts = [], Counter()
    for candidate_index in ordered:
        leaf = catalog.at[candidate_index, LEAF_COLUMN]
        if max_per_leaf is not None and leaf_counts[leaf] >= max_per_leaf:
            continue
        selected.append(int(candidate_index))
        leaf_counts[leaf] += 1
        if len(selected) == top_k:
            break

    columns = ["asin", LEAF_COLUMN, ROOM_COLUMN, "description_original", "image_path"]
    result = catalog.loc[selected, columns].copy()
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result.insert(1, "similarity", [score_by_index[index] for index in selected])
    result.insert(2, "mode", mode)
    result["query_asin"] = str(asin)
    return result.reset_index(drop=True)

def compare_modes(asin, top_k=TOP_K, max_per_leaf=MAX_PER_LEAF):
    image_results = retrieve_complements(asin, "image_to_image", top_k, max_per_leaf)
    text_results = retrieve_complements(asin, "text_to_text", top_k, max_per_leaf)
    overlap = sorted(set(image_results["asin"]) & set(text_results["asin"]))
    summary = pd.DataFrame([
        {
            "mode": "image_to_image",
            "mean_similarity": image_results["similarity"].mean(),
            "distinct_leaf_categories": image_results[LEAF_COLUMN].nunique(),
        },
        {
            "mode": "text_to_text",
            "mean_similarity": text_results["similarity"].mean(),
            "distinct_leaf_categories": text_results[LEAF_COLUMN].nunique(),
        },
    ])
    summary["top_k_overlap"] = len(overlap)
    return summary, image_results, text_results, overlap

In [ ]:
def load_product_image(image_path, size=(420, 420)):
    path = Path(image_path)
    if not path.is_absolute():
        path = ROOT / path
    with Image.open(path) as opened:
        image = opened.convert("RGB")
    return ImageOps.contain(image, size)

def show_product(asin):
    query = catalog.iloc[product_index(asin)]
    display(pd.DataFrame({
        "field": ["ASIN", "room", "leaf category", "description"],
        "value": [query["asin"], query[ROOM_COLUMN], query[LEAF_COLUMN], query["description_original"]],
    }))
    plt.figure(figsize=(4, 4))
    plt.imshow(load_product_image(query["image_path"]))
    plt.axis("off")
    plt.title(f"Query: {query[LEAF_COLUMN]}\n{query['asin']}")
    plt.show()

def plot_results(results, title, columns=4):
    rows = int(np.ceil(len(results) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    for axis, (_, row) in zip(axes, results.iterrows()):
        axis.imshow(load_product_image(row["image_path"]))
        axis.set_title(
            f"#{int(row['rank'])}  {row['similarity']:.3f}\n{row[LEAF_COLUMN]}\n{row['asin']}",
            fontsize=9,
        )
        axis.axis("off")
    for axis in axes[len(results):]:
        axis.axis("off")
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

def inspect_complements(asin, top_k=TOP_K, max_per_leaf=MAX_PER_LEAF):
    query = catalog.iloc[product_index(asin)]
    summary, image_results, text_results, overlap = compare_modes(asin, top_k, max_per_leaf)
    show_product(asin)
    display(summary.round(4))
    print("ASIN overlap between the two rankings:", overlap or "None")
    image_table = image_results[["rank", "similarity", "asin", LEAF_COLUMN, "description_original"]].copy()
    text_table = text_results[["rank", "similarity", "asin", LEAF_COLUMN, "description_original"]].copy()
    for table in (image_table, text_table):
        table["description_original"] = table["description_original"].str.slice(0, 180)
    print("Image-similarity ranking")
    display(image_table.round({"similarity": 4}))
    print("Text-similarity ranking")
    display(text_table.round({"similarity": 4}))
    plot_results(
        image_results,
        f"Image similarity complements — {query[LEAF_COLUMN]} within {query[ROOM_COLUMN]}",
    )
    plot_results(
        text_results,
        f"Text similarity complements — {query[LEAF_COLUMN]} within {query[ROOM_COLUMN]}",
    )
    return image_results, text_results

## 4. Inspect three living-room examples

These ASINs are deterministic examples from the held-out catalog. Run each cell and compare the visual and description-based candidates. The exact query category is excluded in every result.

In [ ]:
EXAMPLE_QUERIES = {
    "sofa": "B089Q7ST3D",
    "coffee_table": "B07XB1P8Q6",
    "chair": "B0CBCBJL3R",
}

for name, asin in EXAMPLE_QUERIES.items():
    assert asin in asin_to_index, f"The expected {name} example is missing from the catalog."
    row = catalog.iloc[product_index(asin)]
    print(name, asin, "->", row[ROOM_COLUMN], "/", row[LEAF_COLUMN])

In [ ]:
sofa_image_results, sofa_text_results = inspect_complements(EXAMPLE_QUERIES["sofa"])

In [ ]:
coffee_image_results, coffee_text_results = inspect_complements(EXAMPLE_QUERIES["coffee_table"])

In [ ]:
chair_image_results, chair_text_results = inspect_complements(EXAMPLE_QUERIES["chair"])

## 5. Optional cross-modal inspection

The next cell compares the sofa image with candidate descriptions and the sofa description with candidate images. Cross-modal scores can behave differently from within-modality scores, so compare rankings rather than raw scores across modes.

In [ ]:
query_asin = EXAMPLE_QUERIES["sofa"]
image_to_text_results = retrieve_complements(query_asin, "image_to_text")
text_to_image_results = retrieve_complements(query_asin, "text_to_image")

display(image_to_text_results[["rank", "similarity", "asin", LEAF_COLUMN, "description_original"]])
plot_results(text_to_image_results, "Sofa description → candidate images")

## 6. Try another catalog product

Copy any ASIN from `catalog` into `QUERY_ASIN`. Use `max_per_leaf=None` to see the un-diversified nearest-neighbor ranking.

In [ ]:
QUERY_ASIN = EXAMPLE_QUERIES["coffee_table"]
CUSTOM_TOP_K = 10
CUSTOM_MAX_PER_LEAF = 2  # Change to None for the raw ranking.

custom_image_results, custom_text_results = inspect_complements(
    QUERY_ASIN, top_k=CUSTOM_TOP_K, max_per_leaf=CUSTOM_MAX_PER_LEAF
)

## 7. How to interpret this experiment

Promising behavior means the retrieved products are in the same usage context and look or read as if they could sensibly accompany the query—for example, sofas retrieving ottomans, chairs, coffee tables, or living-room sets.

Watch for three failure modes:

- **Substitute leakage:** a different leaf label may still describe a substitute, such as a chaise lounge for a sofa.
- **Generic-text attraction:** descriptions with similar marketing language can rank highly without real compatibility.
- **Missing compatibility:** similarity does not learn style coordination, size fit, price fit, or products commonly purchased together.

A stronger next version would combine this room constraint with explicit category-pair rules or priors—such as `Sofas & Couches → Coffee Tables, Ottomans, End Tables`—then eventually learn complementarity from co-view, co-cart, or co-purchase behavior.